# SDH Quick Start
데이터 확인부터 Macro F1 평가와 submission 저장까지 실행하는 입문용 노트북입니다.
위에서 아래로 셀을 순서대로 실행하세요.

In [ ]:
from pathlib import Path
import json
import joblib
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

MEMBER = "SDH"
EXPERIMENT_ID = "sdh-notebook-baseline-001"
SEED = 42

def find_project_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "configs" / "baseline.yaml").exists():
            return path
    raise FileNotFoundError("저장소 루트를 찾지 못했습니다. JupyterLab을 저장소 루트에서 실행하세요.")

ROOT = find_project_root(Path.cwd())
RESULT_DIR = ROOT / "experiments" / MEMBER / "results" / EXPERIMENT_ID
RESULT_DIR.mkdir(parents=True, exist_ok=True)
print("project root:", ROOT)
print("result dir:", RESULT_DIR)

In [ ]:
train = pd.read_csv(ROOT / "data" / "raw" / "train.csv")
test = pd.read_csv(ROOT / "data" / "raw" / "test.csv")
submission = pd.read_csv(ROOT / "data" / "raw" / "sample_submission.csv")

print("train:", train.shape)
print("test:", test.shape)
print("classes:", train["SUBCLASS"].nunique())
display(train[["ID", "SUBCLASS"]].head())

In [ ]:
target = "SUBCLASS"
id_column = "ID"
feature_columns = [c for c in train.columns if c not in {target, id_column}]

train_part, valid_part = train_test_split(
    train,
    test_size=0.25,
    random_state=SEED,
    stratify=train[target],
)

def transform(df: pd.DataFrame) -> pd.DataFrame:
    return df[feature_columns].fillna("WT").ne("WT").astype("int8")

x_train = transform(train_part)
x_valid = transform(valid_part)
model = LogisticRegression(max_iter=1000, random_state=SEED)
model.fit(x_train, train_part[target])
valid_pred = model.predict(x_valid)

metrics = {
    "experiment": EXPERIMENT_ID,
    "owner": MEMBER,
    "model": "LogisticRegression",
    "seed": SEED,
    "validation": "StratifiedHoldout(test_size=0.25)",
    "accuracy": float(accuracy_score(valid_part[target], valid_pred)),
    "f1_macro": float(f1_score(valid_part[target], valid_pred, average="macro")),
    "description": "WT binary notebook baseline",
}
metrics

In [ ]:
(RESULT_DIR / "metrics.json").write_text(
    json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8"
)

final_model = LogisticRegression(max_iter=1000, random_state=SEED)
final_model.fit(transform(train), train[target])
submission[id_column] = test[id_column].values
submission[target] = final_model.predict(transform(test))
submission.to_csv(RESULT_DIR / "submission.csv", index=False)
joblib.dump(final_model, RESULT_DIR / "model.joblib")

print("saved:", RESULT_DIR)
display(submission.head())